42) sklearn 전처리 - 변수변환(인코딩 : 범주 → 수치)

1. 범주형 데이터 인코딩

| 방법 | 주로 사용하는 곳 | 예시 | 특징 |
|---|---|---|---|
| **One-Hot Encoding** (`OneHotEncoder`) | 순서가 없는 입력 특성 `X` | 지역: 서울, 부산, 대전 | 범주마다 열을 만들어 해당 범주를 `1`로 표시 |
| **Ordinal Encoding** (`OrdinalEncoder`) | 순서가 있는 입력 특성 `X` | 만족도: 낮음 < 보통 < 높음 | 정해 둔 순서에 맞춰 숫자로 변환 |
| **Label Encoding** (`LabelEncoder`) | 분류 정답 `y` | 구매 결과: 취소, 유지, 반품 | 정답 클래스 이름을 숫자로 표현. 입력 특성에는 무심코 사용하지 않는 것이 좋음 |
| Binary Encoding | 범주가 매우 많은 입력 특성 | 우편번호, 상품 코드 | 범주를 숫자로 바꾼 뒤 2진수 형태의 여러 열로 표현. 보통 별도 라이브러리가 필요 |
| Target Encoding | 범주가 많고 정답과의 관계를 활용하고 싶은 입력 특성 | 상품 종류별 평균 구매율 | 각 범주를 정답의 평균 등으로 표현. 훈련 데이터 누수에 주의 |
| Count / Frequency Encoding | 범주가 많을 때 | 방문한 도시, 상품 코드 | 범주를 등장 횟수나 비율로 표현 |
| Hashing Encoding | 범주가 매우 많거나 새 범주가 계속 생길 때 | 사용자 ID, 검색어 | 고정된 수의 열에 매핑. 서로 다른 범주가 같은 열에 들어갈 수 있음 |
| **텍스트 벡터화** (`CountVectorizer`, `TfidfVectorizer`) | 문장·리뷰·검색어 | "배송이 빠르고 좋아요" | 단어를 기준으로 숫자 특성으로 변환 |

2. 목적별로 인코더 선택

- 순서 없는 입력값: 보통 OneHotEncoder
- 순서 있는 입력값: 순서를 직접 지정하는 OrdinalEncoder
- 지도학습 분류 정답 y: 필요하면 LabelEncoder
- 범주가 아주 많은 열: 빈도 인코딩, 타깃 인코딩 등을 검토
- 문장이나 리뷰: CountVectorizer 또는 TfidfVectorizer

One-Hot Encoding

- 지역처럼 순서가 없는 입력값을 서울=0, 부산=1처럼 바꾸면 모델이 숫자에 순서가 있다고 오해할 수 있다.
- 독립변수 열 X에는 보통 OneHotEncoder를 사용
- OneHotEncoder를 사용하면 하나의 독립변수에 범주의 개수 만큼 독립변수가 늘어난다.

In [1]:
import pandas as pd
from sklearn.preprocessing import OneHotEncoder

df = pd.DataFrame({
    '색상': ['빨강', '파랑', '초록', '빨강', '초록'],
    '가격': [1000, 1500, 2000, 1200, 1800]
})
display(df)

encoder = OneHotEncoder(sparse_output=False) # sparse_output=False 반환값을 numpy타입, True 행렬타입(판다스 호환 안됨)
result = encoder.fit_transform(df[['색상']])
# 인코딩 결과를 파생변수로 추가
df[['색상_빨강', '색상_파랑', '색상_초록']] = result

display(df)

,색상,가격
0,빨강,1000
1,파랑,1500
2,초록,2000
3,빨강,1200
4,초록,1800


,색상,가격,색상_빨강,색상_파랑,색상_초록
0,빨강,1000,1.0,0.0,0.0
1,파랑,1500,0.0,0.0,1.0
2,초록,2000,0.0,1.0,0.0
3,빨강,1200,1.0,0.0,0.0
4,초록,1800,0.0,1.0,0.0


* 자주사용되는 One-Hot Encoding 3가지 API​

1. 클래스 사용 (OneHotEncoder 직접 활용) <br>
scikit-learn의 OneHotEncoder 객체를 직접 생성하여 .fit()과 .transform()을 거치는 표준적인 방식입니다. <br>
특징: 훈련(Train) 데이터의 범주 체계를 기억하므로, 검증/테스트 데이터 변환 시 열 개수가 어긋나는 것을 방지합니다. <br>
장점: handle_unknown='ignore'를 설정하면 학습 때 없던 새로운 카테고리가 나와도 에러 없이 0으로 변환해 매우 안전합니다. <br>
단점: 변환 결과가 기본적으로 NumPy 배열(Array) 형태로 나오므로, 데이터프레임 구조를 유지하려면 별도 변환 과정이 필요합니다. <br>


In [ ]:
from sklearn.preprocessing import OneHotEncoder

encoder = OneHotEncoder(sparse_output=False, handle_unknown='ignore')
X_train_encoded = encoder.fit_transform(X_train[['color']])
X_val_encoded   = encoder.transform(X_val[['color']])

2. 파이프라인 사용 (Pipeline / ColumnTransformer) <br>
전처리(인코딩, 스케일링 등)와 모델 학습(LogisticRegression 등)을 하나의 체인으로 묶어서 실행하는 프로덕션(실전) 표준 방식입니다. <br>
특징: 데이터 분할 후 파이프라인 하나만 실행하면, 내부에서 자동으로 훈련 데이터 기준으로 인코딩이 적용됩니다. <br>
장점: 데이터 누수(Data Leakage)를 원천 차단할 수 있고, 교차 검증(Cross Validation)이나 하이퍼파라미터 튜닝 시 코드가 대폭 간결해집니다. <br>
단점: 초기 구조를 짜는 데 약간의 scikit-learn 문법 숙지가 필요합니다. <br>

In [ ]:
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder
from sklearn.linear_model import LogisticRegression

# 범주형 열에만 OneHotEncoder 적용
preprocessor = ColumnTransformer(
    transformers=[('cat', OneHotEncoder(handle_unknown='ignore'), ['color'])]
)

# 전처리 + 모델 결합
model_pipe = Pipeline([
    ('preprocessor', preprocessor),
    ('classifier', LogisticRegression())
])

# 학습 및 예측 (알아서 Train 기준 fit 후 변환 적용)
model_pipe.fit(X_train, y_train)
y_pred = model_pipe.predict(X_val)

3. 더미 함수 사용 (pd.get_dummies()) <br>
Pandas에서 제공하는 가장 쉽고 빠른 인코딩 방식입니다. <br>
특징: 데이터프레임을 통째로 넣어 단 한 줄로 빠르게 원-핫 인코딩을 수행합니다. <br>
장점: 사용이 매우 간편하고 결과가 바로 판다스 데이터프레임 형태로 반환되어 탐색적 데이터 분석(EDA)이나 빠른 시각화에 유용합니다. <br>
단점: 머신러닝 전처리용으로는 위험합니다. 전체 데이터를 한 번에 넣으면 데이터 누수가 발생하고, Train/Val을 따로 나누어 적용하면 등장하는 카테고리 종류 차이에 의해 열(Column) 개수가 서로 다르게 맞춰져 모델 예측 시 에러가 발생합니다.

In [ ]:
import pandas as pd

# EDA 및 단일 데이터셋 확인용으로 권장
df_encoded = pd.get_dummies(df, columns=['color'], dtype=int)

2)  Ordinal Encoding

- 만족도를 가진 열값이 순서가 있다면

In [2]:
import pandas as pd
from sklearn.preprocessing import OrdinalEncoder

df = pd.DataFrame({
    '만족도': ['높음', '낮음', '보통', '높음', '보통'],
    '가격': [1000, 1500, 2000, 1200, 1800]
})
display(df)

encoder = OrdinalEncoder(
    categories=[['낮음', '보통', '높음']]
)
# 인코딩 후 파생변수 생성
df['만족도_인코딩'] = encoder.fit_transform(df[['만족도']])

display(df)

,만족도,가격
0,높음,1000
1,낮음,1500
2,보통,2000
3,높음,1200
4,보통,1800


,만족도,가격,만족도_인코딩
0,높음,1000,2.0
1,낮음,1500,0.0
2,보통,2000,1.0
3,높음,1200,2.0
4,보통,1800,1.0


3) Label Encoding

- 분류 정답 열 y의 클래스 이름을 숫자로 바꿀 때 LabelEncoder를 사용한다.
- LabelEncoder는 입력 특성 X의 범주형 열을 바꾸는 용도는 아니다.
- OrdinalEncoder와 달리 LabelEncoder가 자동으로 정한 숫자에는 순서의 의미가 없다.
- 순서가 없지만 모델에 따라 순서나 거리로 해석할 수 있기때문에 X(변수)열에는 One-Hot Encoding을 사용

In [1]:
import pandas as pd
from sklearn.preprocessing import LabelEncoder

df = pd.DataFrame({
    '길이': [30, 35, 40, 45, 50],
    '무게': [300, 350, 400, 450, 500],
    '종류': ['Bream', 'Smelt', 'Bream', 'Smelt', 'Bream']
})
display(df)

encoder = LabelEncoder()
# 정답 열을 숫자로 변환하여 파생변수 생성
df['종류_인코딩'] = encoder.fit_transform(df['종류'])

display(df)

,길이,무게,종류
0,30,300,Bream
1,35,350,Smelt
2,40,400,Bream
3,45,450,Smelt
4,50,500,Bream


,길이,무게,종류,종류_인코딩
0,30,300,Bream,0
1,35,350,Smelt,1
2,40,400,Bream,0
3,45,450,Smelt,1
4,50,500,Bream,0


43) sklearn 전처리 - 변수변환(이산화 : 수치형 → 범주형)

1. 이산화

- 연속적인 수치값을 구간으로 나누는 것

df["나이"] = [15, 23, 35, 47, 62, ...7, 10, 55, ...] <br>

- 구간으로 나누고 범주명(범주화)을 할당 <br>
0~19세 → 청소년 <br>
20~39세 → 청년 <br>
40~59세 → 중년 <br>
60세 이상 → 노년 <br>

<br>

- 이산화+범주화 결과
15 → 청소년 <br>
23 → 청년 <br>
35 → 청년 <br>
47 → 중년 <br>
62 → 노년 <br>
...

① 이산화(구간화) <br>
15 → 0~20 <br>
25 → 20~40 <br>
45 → 40~60 <br>
70 → 60~100 <br>

② 범주화 <br>
0~20   → 청소년 <br>
20~40  → 청년 <br>
40~60  → 중년 <br> 
60~100 → 노년 <br>

③ category dtype <br>
df["나이대"].dtype <br>

* Pandas의 cut()함수를 이용한 이산화(cut()은 기본적으로 category dtype을 반환)



2. sklearn의 KBinsDiscretizer클래스를 이용한 이산화 실습

In [2]:
import pandas as pd
from sklearn.preprocessing import KBinsDiscretizer

# 데이터
df = pd.DataFrame({
    "나이": [15, 23, 35, 47, 62]
})

display(df)

# 이산화
kb = KBinsDiscretizer(
    n_bins=3,              # 3개 구간
    encode="ordinal",      # 구간 번호를 숫자로 표시
    strategy="uniform"    # 동일한 간격으로 구간 분할
)
# 매개변수 strategy 3가지
# uniform   → 구간의 폭을 동일하게
# quantile  → 데이터 개수를 비슷하게
# kmeans    → K-Means 방식으로 구간 결정


df["나이_구간"] = kb.fit_transform(df[["나이"]])

display(df)

,나이
0,15
1,23
2,35
3,47
4,62


,나이,나이_구간
0,15,0.0
1,23,0.0
2,35,1.0
3,47,2.0
4,62,2.0
